In [ ]:
import os
import glob
import random
import cv2
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import timm

In [ ]:
# 1. Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# 2. Aggressive Social Media Compression Function
def apply_social_media_compression(img_bgr):
    """
    Simulates aggressive social media pipeline: 
    Downscale -> Chroma Subsampling (JPEG) -> Gaussian Blur -> Upscale
    """
    h, w, _ = img_bgr.shape
    
    # Step 1: Aggressive Downscaling (mimicking platform resizing)
    scale_factor = random.uniform(0.8, 1.0)
    new_h, new_w = int(h * scale_factor), int(w * scale_factor)
    img_resized = cv2.resize(img_bgr, (new_w, new_h), interpolation=cv2.INTER_AREA)
    img_resized = cv2.resize(img_resized, (w, h), interpolation=cv2.INTER_LINEAR)
    
    # Step 2: Apply a mild Gaussian Blur
    if random.random() < 0.7:
        img_resized = cv2.GaussianBlur(img_resized, (5, 5), 0)
        
    # Step 3: Extreme JPEG Compression with Chroma Subsampling
    quality = random.randint(10, 25)
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    _, encimg = cv2.imencode('.jpg', img_resized, encode_param)
    decimg_bgr = cv2.imdecode(encimg, 1)
    
    # Convert to RGB for PIL / PyTorch
    return cv2.cvtColor(decimg_bgr, cv2.COLOR_BGR2RGB)

In [ ]:
# 3. Balanced Dataset Class with Compression Support
class BalancedDeepfakeDataset(Dataset):
    def __init__(self, split='train', base_dir='/kaggle/input', transform=None, seed=42, compression_prob=0.5):
        self.transform = transform
        self.compression_prob = compression_prob
        self.image_paths = []
        self.labels = []
        self.rng = random.Random(seed)
        
        self._temp_data = {
            'celeb': {0: [], 1: []},
            '140k': {0: [], 1: []},
            'ffpp': {0: [], 1: []}
        }
        
        print(f"[{split.upper()}] Scanning {base_dir} to find datasets...")
        dataset_paths = self._auto_discover_paths(base_dir)
        
        self._load_celebdf(dataset_paths['celeb'], split)
        self._load_140k(dataset_paths['140k'], split)
        self._load_ffpp(dataset_paths['ffpp'], split)
        
        self._balance_and_merge(split)

    def _auto_discover_paths(self, base_dir):
        paths = {'celeb': None, 'ffpp': None, '140k': None}
        for root, dirs, files in os.walk(base_dir):
            if 'Celeb_V2' in dirs and not paths['celeb']:
                paths['celeb'] = os.path.join(root, 'Celeb_V2')
            if 'dataset_processed_split' in dirs and not paths['ffpp']:
                paths['ffpp'] = os.path.join(root, 'dataset_processed_split')
            if 'real-vs-fake' in dirs and not paths['140k']:
                paths['140k'] = os.path.join(root, 'real-vs-fake')
        return paths

    def _add_from_folder(self, dataset_name, folder_path, label):
        if os.path.exists(folder_path):
            for ext in ('*.png', '*.jpg', '*.jpeg'):
                paths = glob.glob(os.path.join(folder_path, ext))
                self._temp_data[dataset_name][label].extend(paths)

    def _load_celebdf(self, base_path, split):
        split_map = {'train': 'Train', 'val': 'Val', 'test': 'Test'}
        target_split = split_map.get(split)
        if base_path and os.path.exists(base_path) and target_split:
            self._add_from_folder('celeb', os.path.join(base_path, target_split, 'real'), 0)
            self._add_from_folder('celeb', os.path.join(base_path, target_split, 'fake'), 1)

    def _load_140k(self, base_path, split):
        split_map = {'train': 'train', 'val': 'valid', 'test': 'test'}
        target_split = split_map.get(split)
        if base_path and os.path.exists(base_path) and target_split:
            self._add_from_folder('140k', os.path.join(base_path, target_split, 'real'), 0)
            self._add_from_folder('140k', os.path.join(base_path, target_split, 'fake'), 1)

    def _load_ffpp(self, base_path, split):
        if base_path:
            split_path = os.path.join(base_path, split)
            if os.path.exists(split_path):
                for class_dir in os.listdir(split_path):
                    dir_path = os.path.join(split_path, class_dir)
                    if os.path.isdir(dir_path):
                        label = 0 if class_dir == 'Real' else 1
                        self._add_from_folder('ffpp', dir_path, label)

    def _balance_and_merge(self, split):
        min_count = float('inf')
        for ds_name, labels in self._temp_data.items():
            count_real = len(labels[0])
            count_fake = len(labels[1])
            if count_real < min_count: min_count = count_real
            if count_fake < min_count: min_count = count_fake
                    
        if min_count == 0:
            print(f"❌ ERROR: One or more categories in {split.upper()} has 0 images. Cannot balance.")
            return

        print(f"[{split.upper()}] Balancing to {min_count} images per category...")
        
        for ds_name, labels in self._temp_data.items():
            for label, paths in labels.items():
                sampled_paths = self.rng.sample(paths, min_count)
                self.image_paths.extend(sampled_paths)
                self.labels.extend([label] * min_count)
        
        combined = list(zip(self.image_paths, self.labels))
        if not combined: return

        self.rng.shuffle(combined)
        self.image_paths, self.labels = zip(*combined)
        self.image_paths = list(self.image_paths)
        self.labels = list(self.labels)
        
        print(f"✅ Final {split} dataset size: {len(self.image_paths)} images ready.\n")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        
        # Load image via CV2 to match the compression pipeline
        img_bgr = cv2.imread(img_path)
        
        # Dynamically apply our compression function
        if self.rng.random() < self.compression_prob:
            img_rgb = apply_social_media_compression(img_bgr)
        else:
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            
        # Convert to PIL for Torchvision transforms
        img_pil = Image.fromarray(img_rgb)
        
        if self.transform:
            img_tensor = self.transform(img_pil)
            
        return img_tensor, torch.tensor(label, dtype=torch.long)

In [ ]:
# 4. Define Transforms for Xception (299x299)
def get_data_transforms(img_size=299):
    norm_mean = [0.485, 0.456, 0.406]
    norm_std = [0.229, 0.224, 0.225]

    train_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomCrop(img_size, padding=4, padding_mode='reflect'),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
        transforms.ToTensor(),
        transforms.Normalize(mean=norm_mean, std=norm_std)
    ])

    val_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=norm_mean, std=norm_std)
    ])

    return train_transform, val_transform

In [ ]:
# 5. Initialize Datasets and DataLoaders
train_transform, val_transform = get_data_transforms(img_size=299)

# Set compression probability to 0.7 for training, 0.0 for validation/testing
train_dataset = BalancedDeepfakeDataset(split='train', transform=train_transform, seed=42, compression_prob=0.7)
val_dataset   = BalancedDeepfakeDataset(split='val', transform=val_transform, seed=42, compression_prob=0.0)
test_dataset  = BalancedDeepfakeDataset(split='test', transform=val_transform, seed=42, compression_prob=0.0)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

In [ ]:
# 6. Initialize Xception Model
print("Loading Xception model...")
# Create pre-trained Xception model and change output features to 2 (Real/Fake)
model = timm.create_model('xception', pretrained=True, num_classes=2)
model = model.to(device)

In [ ]:
# Quick verification test
images, labels = next(iter(train_loader))
images = images.to(device)
outputs = model(images)
print(f"Batch images shape: {images.shape} (Expected: [32, 3, 299, 299])")
print(f"Model outputs shape: {outputs.shape} (Expected: [32, 2])")

In [ ]:
# Fetch one batch to verify
images, labels = next(iter(train_loader))
print(f"Batch images tensor shape: {images.shape}")
print(f"Batch labels: {labels}")

In [ ]:
import torchvision
import matplotlib.pyplot as plt
import numpy as np
import torch.nn as nn
import timm

In [ ]:
def imshow(inp, title=None):
    """Imshow for Tensor."""
    inp = inp.numpy().transpose((1, 2, 0))
    
    # Un-normalize 
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    inp = std * inp + mean
    inp = np.clip(inp, 0, 1)
    
    plt.figure(figsize=(12, 8))
    plt.imshow(inp)
    if title is not None:
        plt.title(title)
    plt.axis('off')
    plt.show()

In [ ]:
# Grab a single batch
images, labels = next(iter(train_loader))

num_images_to_show = 8
images_to_show = images[:num_images_to_show]
labels_to_show = labels[:num_images_to_show]

out = torchvision.utils.make_grid(images_to_show, nrow=4)
label_mapping = {0: "Real", 1: "Fake"}
titles = [label_mapping[int(label.item())] for label in labels_to_show]

imshow(out, title=f"Sample Labels: {titles}")

In [ ]:
class SpatialDeepfakeDetector(nn.Module):
    def __init__(self, pretrained=True):
        super(SpatialDeepfakeDetector, self).__init__()
        
        # Load Xception
        self.backbone = timm.create_model('xception', pretrained=pretrained)
        
        # Replace the classifier head for Binary Classification (1 output node)
        num_ftrs = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Linear(num_ftrs, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 1) 
        )

    def forward(self, x):
        return self.backbone(x)

In [ ]:
# Initialize the model and move it to the GPU
model = SpatialDeepfakeDetector(pretrained=True)
model = model.to(device)

# Test the model output shape
images = images.to(device) 
with torch.no_grad():
    outputs = model(images)

print(f"Output tensor shape: {outputs.shape}")
print("If you see torch.Size([32, 1]), the Xception model is ready!")

In [ ]:
import torch.optim as optim
from tqdm import tqdm
import copy
from torch.optim.lr_scheduler import ReduceLROnPlateau

In [ ]:
# 1. Define Loss Function
criterion = nn.BCEWithLogitsLoss()

In [ ]:
# 2. Define Optimizer (Split learning rates)
backbone_params = []
classifier_params = []
for name, param in model.named_parameters():
    if "backbone.fc" in name:
        classifier_params.append(param)
    else:
        backbone_params.append(param)

optimizer = optim.AdamW([
    {'params':  backbone_params, 'lr': 1e-5}, # Slow updates to features
    {'params': classifier_params, 'lr': 1e-3}       # Fast updates to the head
])

In [ ]:
# 3. Early Stopping Class
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0, verbose=True):
        self.patience = patience
        self.min_delta = min_delta
        self.verbose = verbose
        self.counter = 0
        self.best_loss = float('inf')
        self.early_stop = False
        self.best_model_wts = None

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            if self.verbose:
                print(f"Validation loss decreased ({self.best_loss:.4f} --> {val_loss:.4f}). Saving model weights...")
            self.best_loss = val_loss
            self.best_model_wts = copy.deepcopy(model.state_dict())
            self.counter = 0
        else:
            self.counter += 1
            print(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True

In [ ]:
# ART FGSM setup (added): installs/imports ART and creates a two-logit adapter for the existing one-logit model.
# The original model, dataset, transforms, loss, and optimizer definitions remain unchanged.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("art") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "adversarial-robustness-toolbox"])

from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import FastGradientMethod
from torch.utils.data import Subset, DataLoader
import numpy as np
import gc

# ART's classifier interface expects one output per class. This adapter converts the
# existing single BCE logit z into two class logits [-z, z] without changing model weights.
class ARTTwoLogitAdapter(nn.Module):
    def __init__(self, binary_model):
        super().__init__()
        self.binary_model = binary_model

    def forward(self, x):
        z = self.binary_model(x)
        return torch.cat((-z, z), dim=1)

# Match the normalization in get_data_transforms(). ART receives raw [0,1] images
# and applies this normalization before calling the model.
ART_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(1, 3, 1, 1)
ART_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(1, 3, 1, 1)

def normalized_to_raw(x):
    mean = torch.as_tensor(ART_MEAN[0], device=x.device, dtype=x.dtype)
    std = torch.as_tensor(ART_STD[0], device=x.device, dtype=x.dtype)
    return (x * std + mean).clamp(0.0, 1.0)

def raw_to_normalized(x):
    mean = torch.as_tensor(ART_MEAN[0], device=x.device, dtype=x.dtype)
    std = torch.as_tensor(ART_STD[0], device=x.device, dtype=x.dtype)
    return (x - mean) / std

# Use a 20% training subset to keep FGSM training feasible in a 12-hour Kaggle session.
# Validation and test datasets remain unchanged and are evaluated in full.
ART_TRAIN_FRACTION = 0.20
_art_rng = np.random.default_rng(42)
_art_n = max(1, int(len(train_dataset) * ART_TRAIN_FRACTION))
_art_indices = _art_rng.choice(len(train_dataset), size=_art_n, replace=False).tolist()
art_train_dataset = Subset(train_dataset, _art_indices)
art_train_loader = DataLoader(
    art_train_dataset, batch_size=train_loader.batch_size, shuffle=True,
    num_workers=train_loader.num_workers, pin_memory=torch.cuda.is_available(),
    persistent_workers=(train_loader.num_workers > 0)
)

art_model = ARTTwoLogitAdapter(model).to(device)
art_classifier = PyTorchClassifier(
    model=art_model,
    loss=nn.CrossEntropyLoss(),
    optimizer=optimizer,
    input_shape=(3, 299, 299),
    nb_classes=2,
    clip_values=(0.0, 1.0),
    preprocessing=(ART_MEAN, ART_STD),
    device_type="gpu" if torch.cuda.is_available() else "cpu"
)

# FGSM epsilon is specified in raw pixel space [0,1].
FGSM_EPSILON = 8.0 / 255.0
attack_fgsm = FastGradientMethod(
    estimator=art_classifier, eps=FGSM_EPSILON, norm=np.inf
)
print(f"ART FGSM ready. Training subset: {_art_n:,}/{len(train_dataset):,} images "
      f"({100 * _art_n / len(train_dataset):.1f}%).")


In [ ]:
# ART FGSM adversarial training — 5 epochs, memory-conscious sequential passes.
# Existing model/loss/optimizer definitions are reused. Only the training procedure is changed.
epochs = 5
early_stopper = EarlyStopping(patience=2, verbose=True)
scheduler = ReduceLROnPlateau(optimizer, mode="min", patience=1, factor=0.1)

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    clean_correct = 0
    clean_total = 0
    loop = tqdm(art_train_loader, total=len(art_train_loader),
                desc=f"ART FGSM Epoch {epoch + 1}/{epochs}", miniters=25, leave=True)

    for inputs, labels in loop:
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device).long().view(-1)

        # Generate FGSM examples in raw [0,1] space through ART, using the same
        # ImageNet normalization as the original model's transforms.
        model.eval()
        raw_inputs = normalized_to_raw(inputs)
        x_np = raw_inputs.detach().cpu().numpy().astype(np.float32, copy=False)
        y_np = labels.detach().cpu().numpy().astype(np.int64, copy=False)
        adv_np = attack_fgsm.generate(x=x_np, y=y_np)
        adv_raw = torch.from_numpy(np.asarray(adv_np, dtype=np.float32)).to(device)
        adv_inputs = raw_to_normalized(adv_raw)

        # Sequential backward passes avoid concatenating clean+adversarial batches
        # (the previous run's combined batch caused a CUDA out-of-memory error).
        model.train()
        optimizer.zero_grad(set_to_none=True)

        clean_logits = model(inputs)
        clean_labels = labels.float().view(-1, 1)
        clean_loss = criterion(clean_logits, clean_labels)
        clean_loss_value = clean_loss.item()
        (0.5 * clean_loss).backward()
        clean_preds = (torch.sigmoid(clean_logits.detach()) > 0.5).view(-1).long()
        clean_correct += (clean_preds == labels).sum().item()
        clean_total += labels.size(0)
        del clean_logits, clean_loss, clean_preds, clean_labels

        adv_logits = model(adv_inputs)
        adv_labels = labels.float().view(-1, 1)
        adv_loss = criterion(adv_logits, adv_labels)
        adv_loss_value = adv_loss.item()
        (0.5 * adv_loss).backward()
        optimizer.step()

        batch_loss = 0.5 * (clean_loss_value + adv_loss_value)
        running_loss += batch_loss
        loop.set_postfix(loss=f"{batch_loss:.4f}",
                         clean_acc=f"{100 * clean_correct / max(clean_total, 1):.1f}%")

        del inputs, labels, raw_inputs, x_np, y_np, adv_np, adv_raw, adv_inputs
        del adv_logits, adv_loss, adv_labels
        # Do not call empty_cache()/gc.collect() on every batch; that adds overhead.
        if (loop.n + 1) % 100 == 0:
            gc.collect()

    avg_train_loss = running_loss / max(len(art_train_loader), 1)
    clean_train_accuracy = 100.0 * clean_correct / max(clean_total, 1)

    # Full clean validation; this is the key metric for model selection.
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc="Clean validation", miniters=50, leave=False):
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device).float().view(-1, 1)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            preds = (torch.sigmoid(outputs) > 0.5).float()
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
            del inputs, labels, outputs, loss, preds

    avg_val_loss = val_loss / max(len(val_loader), 1)
    val_accuracy = 100.0 * val_correct / max(val_total, 1)
    print(f"\nEpoch {epoch + 1}/{epochs}")
    print(f"Train loss (clean + FGSM): {avg_train_loss:.4f}")
    print(f"Clean training accuracy: {clean_train_accuracy:.2f}%")
    print(f"Clean validation loss: {avg_val_loss:.4f}")
    print(f"Clean validation accuracy: {val_accuracy:.2f}%")

    # Save a checkpoint after each completed epoch so progress is recoverable.
    torch.save(model.state_dict(), f"xception_art_fgsm_epoch_{epoch + 1}.pth")
    scheduler.step(avg_val_loss)
    early_stopper(avg_val_loss, model)
    if early_stopper.early_stop:
        print(f"Early stopping at epoch {epoch + 1}.")
        break

if early_stopper.best_model_wts is not None:
    model.load_state_dict(early_stopper.best_model_wts)
    torch.save(model.state_dict(), "xception_art_fgsm_best.pth")
print("ART FGSM adversarial training completed.")


In [ ]:
import torch.nn.functional as F

# Unfreeze layers for GradCAM
for param in model.backbone.parameters():
    param.requires_grad = True

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        target_layer.register_forward_hook(self.save_activation)
        target_layer.register_full_backward_hook(self.save_gradient)

    def save_activation(self, module, input, output):
        self.activations = output

    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def generate_heatmap(self, input_tensor):
        model_output = self.model(input_tensor)
        self.model.zero_grad()
        model_output.backward(retain_graph=True)
        
        pooled_gradients = torch.mean(self.gradients, dim=[0, 2, 3])
        activations = self.activations.detach()[0]
        for i in range(activations.shape[0]):
            activations[i, :, :] *= pooled_gradients[i]
            
        heatmap = torch.mean(activations, dim=0).squeeze().cpu().numpy()
        heatmap = np.maximum(heatmap, 0)
        if np.max(heatmap) != 0:
            heatmap /= np.max(heatmap)
        return heatmap

# Target the deepest convolutional layer
target_layer = model.backbone.conv4 
cam = GradCAM(model, target_layer)

model.eval() 
sample_img, sample_label = next(iter(test_loader)) # Use test loader for unseen data
sample_img = sample_img[0:1].to(device)

heatmap = cam.generate_heatmap(sample_img)

# --- VISUALIZATION ---
inv_normalize = transforms.Normalize(
    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
    std=[1/0.229, 1/0.224, 1/0.225]
)
disp_img = inv_normalize(sample_img[0].cpu())
disp_img = disp_img.permute(1, 2, 0).numpy()
disp_img = np.clip(disp_img, 0, 1)

heatmap_resized = cv2.resize(heatmap, (299, 299))
heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
heatmap_colored = np.float32(heatmap_colored) / 255
heatmap_colored = heatmap_colored[..., ::-1] 

overlay = heatmap_colored * 0.5 + disp_img
overlay = np.clip(overlay, 0, 1)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(disp_img)
axes[0].set_title(f"Original (True: {'Fake' if sample_label[0].item() == 1 else 'Real'})")
axes[0].axis('off')

axes[1].imshow(overlay)
axes[1].set_title("Grad-CAM Explanation")
axes[1].axis('off')

plt.show()

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import seaborn as sns

print("Starting Final Evaluation on Test Set...")

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="Testing"):
        inputs = inputs.to(device)
        labels = labels.to(device).view(-1, 1).float()
        
        outputs = model(inputs)
        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).float()
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds, zero_division=0)
rec = recall_score(all_labels, all_preds, zero_division=0)
f1 = f1_score(all_labels, all_preds, zero_division=0)

print("\n--- Final XceptionNet Performance ---")
print(f"Accuracy:  {acc * 100:.2f}%")
print(f"Precision: {prec * 100:.2f}%")
print(f"Recall:    {rec * 100:.2f}%")
print(f"F1-Score:  {f1 * 100:.2f}%")

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Predicted Real', 'Predicted Fake'], 
            yticklabels=['Actually Real', 'Actually Fake'])
plt.title('XceptionNet Deepfake Detection Confusion Matrix')
plt.show()

torch.save(model.state_dict(), 'xception_deepfake_detector_final.pth')
print("Model weights successfully saved!")

In [ ]:
# NEW CELL 22 — Final ART FGSM Adversarial Evaluation

import numpy as np
import torch
from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score
)

model.eval()

fgsm_preds = []
fgsm_labels = []

for inputs, labels in tqdm(test_loader, desc="ART FGSM Testing"):
    labels = labels.long().view(-1)

    # Convert normalized test images to raw [0, 1] pixel space
    inputs = inputs.to(device)
    raw_inputs = normalized_to_raw(inputs)

    x_np = raw_inputs.detach().cpu().numpy().astype(np.float32)
    y_np = labels.cpu().numpy().astype(np.int64)

    # Generate adversarial test images using ART FGSM
    adv_np = attack_fgsm.generate(x=x_np, y=y_np)

    # Convert back to normalized inputs for Xception
    adv_inputs = raw_to_normalized(
        torch.from_numpy(adv_np).float().to(device)
    )

    # Predict on adversarial images
    with torch.no_grad():
        outputs = model(adv_inputs)
        preds = (torch.sigmoid(outputs) > 0.5).long().view(-1)

    fgsm_preds.extend(preds.cpu().numpy().tolist())
    fgsm_labels.extend(labels.cpu().numpy().tolist())

fgsm_acc = accuracy_score(fgsm_labels, fgsm_preds)
fgsm_prec = precision_score(fgsm_labels, fgsm_preds, zero_division=0)
fgsm_rec = recall_score(fgsm_labels, fgsm_preds, zero_division=0)
fgsm_f1 = f1_score(fgsm_labels, fgsm_preds, zero_division=0)

print("\n--- ART FGSM Adversarial Test Performance ---")
print(f"FGSM Accuracy:  {fgsm_acc * 100:.2f}%")
print(f"FGSM Precision: {fgsm_prec * 100:.2f}%")
print(f"FGSM Recall:    {fgsm_rec * 100:.2f}%")
print(f"FGSM F1-Score:  {fgsm_f1 * 100:.2f}%")